In [11]:
import pandas as pd
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, TimeDistributed, Dense
from sklearn.model_selection import train_test_split

In [12]:
df = pd.read_csv("ner_dataset.csv", encoding="latin1")
print(df.head())

    Sentence #           Word  POS Tag
0  Sentence: 1      Thousands  NNS   O
1          NaN             of   IN   O
2          NaN  demonstrators  NNS   O
3          NaN           have  VBP   O
4          NaN        marched  VBN   O


In [13]:
# Fill missing Sentence numbers
df["Sentence #"] = df["Sentence #"].ffill()

# Fill missing words
df["Word"] = df["Word"].bfill()

print(df.isnull().sum())

Sentence #    0
Word          0
POS           0
Tag           0
dtype: int64


In [14]:
sentences = df.groupby("Sentence #")["Word"].apply(list).values
pos_tags = df.groupby("Sentence #")["POS"].apply(list).values

print(sentences[0])
print(pos_tags[0])

['Thousands', 'of', 'demonstrators', 'have', 'marched', 'through', 'London', 'to', 'protest', 'the', 'war', 'in', 'Iraq', 'and', 'demand', 'the', 'withdrawal', 'of', 'British', 'troops', 'from', 'that', 'country', '.']
['NNS', 'IN', 'NNS', 'VBP', 'VBN', 'IN', 'NNP', 'TO', 'VB', 'DT', 'NN', 'IN', 'NNP', 'CC', 'VB', 'DT', 'NN', 'IN', 'JJ', 'NNS', 'IN', 'DT', 'NN', '.']


In [15]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

X = tokenizer.texts_to_sequences(sentences)

In [16]:
tag_tokenizer = Tokenizer()
tag_tokenizer.fit_on_texts(pos_tags)

y = tag_tokenizer.texts_to_sequences(pos_tags)

In [17]:
MAX_LEN = 128

X = pad_sequences(X, maxlen=MAX_LEN, padding='post')
y = pad_sequences(y, maxlen=MAX_LEN, padding='post')

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8
)

In [22]:
num_tags = np.max(y) + 1

model = Sequential()
model.add(Embedding(
    input_dim=len(tokenizer.word_index)+1,
    output_dim=64
))
model.add(LSTM(64, return_sequences=True))
model.add(TimeDistributed(Dense(num_tags, activation='softmax')))

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [23]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

#model.summary()

In [24]:
model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=5
)

Epoch 1/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 65s 50ms/step - accuracy: 0.9519 - loss: 0.2097
Epoch 2/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 60s 50ms/step - accuracy: 0.9927 - loss: 0.0262
Epoch 3/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 61s 51ms/step - accuracy: 0.9945 - loss: 0.0168
Epoch 4/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 61s 50ms/step - accuracy: 0.9954 - loss: 0.0135
Epoch 5/5
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 60s 50ms/step - accuracy: 0.9959 - loss: 0.0115


In [25]:
loss, acc = model.evaluate(X_test, y_test)
print("Accuracy:", acc)

300/300 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9933 - loss: 0.0204
Accuracy: 0.9932789206504822


In [26]:
# Take sample sentence
test_sentence = sentences[0]

# Convert to sequence
test_seq = tokenizer.texts_to_sequences([test_sentence])
test_seq = pad_sequences(test_seq, maxlen=MAX_LEN, padding='post')

# Predict
pred = model.predict(test_seq)

pred = np.argmax(pred, axis=-1)[0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 919ms/step


In [27]:
index_to_tag = {v:k for k,v in tag_tokenizer.word_index.items()}

for word, tag in zip(test_sentence, pred[:len(test_sentence)]):
    print(word, "->", index_to_tag.get(tag))

Thousands -> nns
of -> in
demonstrators -> nns
have -> vbp
marched -> vbn
through -> in
London -> nnp
to -> to
protest -> vb
the -> dt
war -> nn
in -> in
Iraq -> nnp
and -> cc
demand -> nn
the -> dt
withdrawal -> nn
of -> in
British -> jj
troops -> nns
from -> in
that -> dt
country -> nn
. -> .
